In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(
    "../data/processed/cleaned_uhi_v2.csv"
)

df.head()

,Elevation,LST,Latitude,Longitude,NDBI,NDVI,Population
0,919,43.919170,19.970832,75.438700,0.127207,0.162286,1.562392
1,488,39.060455,19.729093,75.209548,-0.272423,0.570751,1.361367
2,505,40.231127,19.788775,75.225639,0.055877,0.099371,15.386780
3,757,44.884761,19.960297,75.544582,0.121404,0.196037,2.643492
4,586,48.502735,19.867709,75.486062,0.133151,0.165262,31.397209


In [4]:
df["Green_Built_Ratio"] = (
    df["NDVI"] /
    (abs(df["NDBI"]) + 0.01)
)

df.head()

,Elevation,LST,Latitude,Longitude,NDBI,NDVI,Population,Green_Built_Ratio
0,919,43.919170,19.970832,75.438700,0.127207,0.162286,1.562392,1.182783
1,488,39.060455,19.729093,75.209548,-0.272423,0.570751,1.361367,2.020910
2,505,40.231127,19.788775,75.225639,0.055877,0.099371,15.386780,1.508423
3,757,44.884761,19.960297,75.544582,0.121404,0.196037,2.643492,1.491866
4,586,48.502735,19.867709,75.486062,0.133151,0.165262,31.397209,1.154459


In [5]:
df["Population_Heat_Index"] = (
    df["Population"] *
    df["NDBI"]
)

In [6]:
df["Elevation_Cooling_Index"] = (
    df["Elevation"] *
    df["NDVI"]
)

In [7]:
df.head()

,Elevation,LST,Latitude,Longitude,NDBI,NDVI,Population,Green_Built_Ratio,Population_Heat_Index,Elevation_Cooling_Index
0,919,43.919170,19.970832,75.438700,0.127207,0.162286,1.562392,1.182783,0.198747,149.140825
1,488,39.060455,19.729093,75.209548,-0.272423,0.570751,1.361367,2.020910,-0.370867,278.526341
2,505,40.231127,19.788775,75.225639,0.055877,0.099371,15.386780,1.508423,0.859774,50.182390
3,757,44.884761,19.960297,75.544582,0.121404,0.196037,2.643492,1.491866,0.320931,148.400233
4,586,48.502735,19.867709,75.486062,0.133151,0.165262,31.397209,1.154459,4.180581,96.843708


In [8]:
df.columns

Index(['Elevation', 'LST', 'Latitude', 'Longitude', 'NDBI', 'NDVI',
       'Population', 'Green_Built_Ratio', 'Population_Heat_Index',
       'Elevation_Cooling_Index'],
      dtype='str')

In [9]:
df[
    [
        "Green_Built_Ratio",
        "Population_Heat_Index",
        "Elevation_Cooling_Index"
    ]
].describe()

,Green_Built_Ratio,Population_Heat_Index,Elevation_Cooling_Index
count,9893.000000,9893.000000,9893.000000
mean,3.984625,0.384507,148.084978
std,4.301744,2.412859,68.702984
min,-2.903357,-32.066005,-71.859134
25%,1.500314,-0.078914,101.100586
50%,2.477985,0.164021,133.644636
75%,4.535040,0.358065,182.240664
max,46.586742,47.796099,505.259259


In [10]:
df.to_csv(
    "../data/processed/featured_uhi_v2.csv",
    index=False
)

print("V2 Feature Engineering Complete")

V2 Feature Engineering Complete


In [11]:
df.shape

(9893, 10)

# Dataset V3 Feature Engineering

This section prepares Dataset V3 for machine learning by one-hot encoding the categorical `LandCover` feature.

Latitude and Longitude are retained only for mapping and spatial block cross-validation. They are excluded from model features to avoid spatial leakage.

In [1]:
import pandas as pd
import numpy as np
import os

df_v3 = pd.read_csv("../data/processed/cleaned_uhi_v3.csv")

print("Dataset V3 loaded successfully")
print("Shape:", df_v3.shape)
display(df_v3.head())

Dataset V3 loaded successfully
Shape: (9893, 8)


,Elevation,LST,LandCover,Latitude,Longitude,NDBI,NDVI,Population
0,919,43.919170,40,19.970832,75.438700,0.127207,0.162286,1.562392
1,488,39.060455,40,19.729093,75.209548,-0.272423,0.570751,1.361367
2,505,40.231127,30,19.788775,75.225639,0.055877,0.099371,15.386780
3,757,44.884761,40,19.960297,75.544582,0.121404,0.196037,2.643492
4,586,48.502735,30,19.867709,75.486062,0.133151,0.165262,31.397209


In [2]:
landcover_labels = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up land",
    60: "Bare_sparse_vegetation",
    80: "Permanent_water_bodies"
}

df_v3["LandCover_Name"] = df_v3["LandCover"].map(landcover_labels)

print("LandCover classes:")
print(df_v3[["LandCover", "LandCover_Name"]].drop_duplicates().sort_values("LandCover"))

LandCover classes:
     LandCover          LandCover_Name
29          10              Tree cover
9           20               Shrubland
2           30               Grassland
0           40                Cropland
19          50           Built-up land
77          60  Bare_sparse_vegetation
626         80  Permanent_water_bodies


In [3]:
landcover_dummies = pd.get_dummies(
    df_v3["LandCover_Name"],
    prefix="LandCover",
    dtype=int
)

df_v3_features = pd.concat(
    [df_v3, landcover_dummies],
    axis=1
)

print("One-hot encoded LandCover columns:")
print(landcover_dummies.columns.tolist())

print("\nFeature-engineered shape:", df_v3_features.shape)

One-hot encoded LandCover columns:
['LandCover_Bare_sparse_vegetation', 'LandCover_Built-up land', 'LandCover_Cropland', 'LandCover_Grassland', 'LandCover_Permanent_water_bodies', 'LandCover_Shrubland', 'LandCover_Tree cover']

Feature-engineered shape: (9893, 16)


In [4]:
base_features_v3 = [
    "NDVI",
    "NDBI",
    "Elevation",
    "Population"
]

landcover_feature_columns = landcover_dummies.columns.tolist()

model_features_v3 = base_features_v3 + landcover_feature_columns

target_v3 = "LST"

spatial_columns_v3 = ["Latitude", "Longitude"]

print("Model features:")
print(model_features_v3)

print("\nTarget:")
print(target_v3)

print("\nSpatial-only columns, excluded from prediction:")
print(spatial_columns_v3)

Model features:
['NDVI', 'NDBI', 'Elevation', 'Population', 'LandCover_Bare_sparse_vegetation', 'LandCover_Built-up land', 'LandCover_Cropland', 'LandCover_Grassland', 'LandCover_Permanent_water_bodies', 'LandCover_Shrubland', 'LandCover_Tree cover']

Target:
LST

Spatial-only columns, excluded from prediction:
['Latitude', 'Longitude']


In [6]:
model_dataset_v3 = df_v3_features[
    model_features_v3 + [target_v3] + spatial_columns_v3
].copy()

print("Final Dataset V3 model shape:", model_dataset_v3.shape)

print("\nMissing values:")
print(model_dataset_v3.isnull().sum())

display(model_dataset_v3.head())

Final Dataset V3 model shape: (9893, 14)

Missing values:
NDVI                                0
NDBI                                0
Elevation                           0
Population                          0
LandCover_Bare_sparse_vegetation    0
LandCover_Built-up land             0
LandCover_Cropland                  0
LandCover_Grassland                 0
LandCover_Permanent_water_bodies    0
LandCover_Shrubland                 0
LandCover_Tree cover                0
LST                                 0
Latitude                            0
Longitude                           0
dtype: int64


,NDVI,NDBI,Elevation,Population,LandCover_Bare_sparse_vegetation,LandCover_Built-up land,LandCover_Cropland,LandCover_Grassland,LandCover_Permanent_water_bodies,LandCover_Shrubland,LandCover_Tree cover,LST,Latitude,Longitude
0,0.162286,0.127207,919,1.562392,0,0,1,0,0,0,0,43.919170,19.970832,75.438700
1,0.570751,-0.272423,488,1.361367,0,0,1,0,0,0,0,39.060455,19.729093,75.209548
2,0.099371,0.055877,505,15.386780,0,0,0,1,0,0,0,40.231127,19.788775,75.225639
3,0.196037,0.121404,757,2.643492,0,0,1,0,0,0,0,44.884761,19.960297,75.544582
4,0.165262,0.133151,586,31.397209,0,0,0,1,0,0,0,48.502735,19.867709,75.486062


In [7]:
print("One-hot encoding row-sum check:")

row_sums = model_dataset_v3[landcover_feature_columns].sum(axis=1)

print(row_sums.value_counts().sort_index())

if (row_sums == 1).all():
    print("\nValidation passed: every row belongs to exactly one LandCover class.")
else:
    print("\nWarning: some rows do not have exactly one LandCover class.")

One-hot encoding row-sum check:
1    9893
Name: count, dtype: int64

Validation passed: every row belongs to exactly one LandCover class.


In [8]:
feature_engineering_summary_v3 = pd.DataFrame({
    "Item": [
        "Dataset Version",
        "Rows",
        "Base Numeric Features",
        "LandCover Feature",
        "One-Hot Encoded LandCover Columns",
        "Total Model Features",
        "Target Variable",
        "Excluded Prediction Features"
    ],
    "Value": [
        "Dataset V3",
        len(model_dataset_v3),
        ", ".join(base_features_v3),
        "LandCover",
        ", ".join(landcover_feature_columns),
        len(model_features_v3),
        target_v3,
        ", ".join(spatial_columns_v3)
    ]
})

feature_engineering_summary_v3

,Item,Value
0,Dataset Version,Dataset V3
1,Rows,9893
2,Base Numeric Features,"NDVI, NDBI, Elevation, Population"
3,LandCover Feature,LandCover
4,One-Hot Encoded LandCover Columns,"LandCover_Bare_sparse_vegetation, LandCover_Bu..."
5,Total Model Features,11
6,Target Variable,LST
7,Excluded Prediction Features,"Latitude, Longitude"


In [9]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../outputs/reports", exist_ok=True)

df_v3_features.to_csv(
    "../data/processed/featured_uhi_v3.csv",
    index=False
)

model_dataset_v3.to_csv(
    "../data/processed/model_ready_uhi_v3.csv",
    index=False
)

feature_engineering_summary_v3.to_csv(
    "../outputs/reports/feature_engineering_summary_v3.csv",
    index=False
)

print("Saved:")
print("data/processed/featured_uhi_v3.csv")
print("data/processed/model_ready_uhi_v3.csv")
print("outputs/reports/feature_engineering_summary_v3.csv")

Saved:
data/processed/featured_uhi_v3.csv
data/processed/model_ready_uhi_v3.csv
outputs/reports/feature_engineering_summary_v3.csv


## Dataset V3 Feature Engineering Conclusion

Dataset V3 was prepared for machine learning by retaining the continuous physical drivers NDVI, NDBI, Elevation, and Population, and one-hot encoding the categorical `LandCover` feature.

The model-ready Dataset V3 contains the target variable `LST`, the model feature set, and Latitude/Longitude for spatial grouping only. Latitude and Longitude are excluded from prediction features to prevent the model from memorizing location-based heat patterns.

Dataset V3 will be evaluated against Dataset V2 using the same leakage-aware spatial block cross-validation procedure and the same R², MAE, and RMSE metrics.